<div align="center">

# 📉 Semana 3 · Derivadas, gradiente y probabilidad

**Máster en Creación de Modelos de IA · Nivel I: «Construye Gleihar»** · SagaTechSchool Pro

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saga1555/SagaEstudios7316/blob/main/semana03/semana03_gradiente_probabilidad.ipynb)

</div>

---

En el capítulo 3 viste **de dónde sale el gradiente** (derivadas y regla de la cadena), **cómo se usa** (descenso de gradiente) y **por qué medimos el error con la entropía cruzada** (probabilidad y máxima verosimilitud). En este cuaderno lo compruebas todo con PyTorch, y Gleihar deja de aprender contando: **aprende con gradiente**.

### 🎯 Al terminar podrás

1. Aproximar derivadas con la **diferencia central** y medir su error.
2. Calcular la **regla de la cadena** a mano y verificarla con `backward()`.
3. Dibujar **curvas de nivel** y el gradiente de una pérdida.
4. Escribir el **ciclo de entrenamiento** y elegir la **tasa de aprendizaje**.
5. Simular probabilidades, la **normal** y la **regla de Bayes**.
6. Programar la **entropía cruzada** y entrenar a Gleihar con gradiente.

### 🗺️ Mapa del cuaderno

| Parte | Qué haces | Sección del libro | Tiempo |
|---|---|---|---|
| 0 · Tu entorno | Compruebas tu GPU | 1.7 | 5 min |
| 1 · Derivadas numéricas | Diferencia central y secantes | 3.1 | 15 min |
| 2 · Regla de la cadena | La neurona a mano y con `backward()` | 3.2 y 3.5 | 20 min |
| 3 · Gradiente | Curvas de nivel de la pérdida de los terrenos | 3.3 | 15 min |
| 4 · Descenso de gradiente | Tu ciclo de 5 pasos y la tasa de aprendizaje | 3.4 y 3.5 | 25 min |
| 5 · Probabilidad | Monedas, normales y Bayes por simulación | 3.6 | 20 min |
| 6 · Entropía cruzada | Tu función de pérdida y la verosimilitud | 3.7 | 20 min |
| 7 · Gleihar aprende con gradiente | Logits, softmax y entrenamiento | 3.7 | 20 min |
| 8 · Entrega | Generas tu reporte | — | 5 min |

| Símbolo | Significa |
|---|---|
| 🔮 **Predice** | Antes de ejecutar, escribe qué crees que pasará. |
| ✍️ **Tu turno** | Completa el código donde dice `...`. |
| ✅ **Comprueba** | Lee la explicación después de ejecutar. |
| ⚠️ **Cuidado** | Error frecuente y cómo evitarlo. |

> 💡 Guarda tu copia con *Archivo → Guardar una copia en Drive*. Este cuaderno funciona igual con o sin GPU.

## 0️⃣ Tu entorno

In [ ]:
import sys, json, time, math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

EN_COLAB = "google.colab" in sys.modules
GLEIHAR = {"semana": 3, "entorno": "Google Colab" if EN_COLAB else "Equipo local",
           "pytorch": torch.__version__, "dispositivo": "cuda" if torch.cuda.is_available() else "cpu"}
if torch.cuda.is_available():
    GLEIHAR["gpu"] = torch.cuda.get_device_name(0)
torch.manual_seed(2026)
print(json.dumps(GLEIHAR, ensure_ascii=False, indent=2))

## 1️⃣ Derivadas numéricas (sección 3.1)

$$f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}\qquad\text{(diferencia central, error}\approx h^2)$$

### ✍️ Tu turno: programa la diferencia central.

In [ ]:
def derivada_central(f, x, h=1e-3):
    """Aproxima f'(x) con la diferencia central."""
    ...

In [ ]:
sigmoide = lambda x: 1 / (1 + math.exp(-x))
def revisar_parte1():
    try:
        d1 = derivada_central(lambda x: x**2, 3.0)
        exacta = sigmoide(2) * (1 - sigmoide(2))          # σ'(x) = σ(x)(1 − σ(x))
        print(f"d/dx x² en 3      → {d1:.6f}  (exacta: 6)")
        print(f"σ'(2) numérica    → {derivada_central(sigmoide, 2.0):.6f}  (exacta: {exacta:.6f})")
        print("\n   h      adelante     error       central      error")
        for h in [0.1, 0.01, 0.001]:
            ad = (sigmoide(2 + h) - sigmoide(2)) / h
            ce = derivada_central(sigmoide, 2.0, h)
            print(f"{h:6}  {ad:.6f}  {abs(ad-exacta):.1e}   {ce:.6f}  {abs(ce-exacta):.1e}")
        ok = abs(d1 - 6) < 1e-6 and abs(derivada_central(sigmoide, 2.0) - exacta) < 1e-6
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ derivada central" if ok else "❌ derivada central: devuelve (f(x+h) − f(x−h)) / (2h)")
    GLEIHAR["parte1"] = "1/1" if ok else "0/1"
revisar_parte1()

> 🔮 **Predice:** ¿qué pasa con las secantes de $x^2$ en $x=3$ cuando $h$ se hace pequeño? Ejecuta y compara con la figura 3.1 del libro.

In [ ]:
xs = torch.linspace(0.5, 4.5, 100)
plt.figure(figsize=(6, 4))
plt.plot(xs, xs**2, "k", lw=2, label="$x^2$")
for h, color in [(1, "gray"), (0.5, "tab:blue"), (0.1, "tab:green")]:
    m = ((3 + h)**2 - 9) / h
    plt.plot(xs, 9 + m * (xs - 3), "--", color=color, label=f"secante h={h}: pendiente {m:.1f}")
plt.plot(xs, 9 + 6 * (xs - 3), color="tab:orange", lw=2, label="tangente: pendiente 6")
plt.ylim(-1, 20); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("Las secantes se acercan a la tangente"); plt.show()

## 2️⃣ La regla de la cadena, a mano y con PyTorch (secciones 3.2 y 3.5)

La neurona del ejemplo: $x=2$, $y=1$, $w=0{,}5$, $b=-1$.

$$z=wx+b,\quad a=\sigma(z),\quad L=(a-y)^2$$

### ✍️ Tu turno: completa el recorrido hacia atrás.
Cada derivada es **la que llega desde atrás × la derivada local**.

In [ ]:
x, y, w, b = 2.0, 1.0, 0.5, -1.0
# Hacia adelante
z = w * x + b
a = sigmoide(z)
L = (a - y) ** 2
# Hacia atrás
dL_da = 2 * (a - y)
dL_dz = ...            # dL_da por la derivada de la sigmoide: a(1 − a)
dL_dw = ...            # dL_dz por dz/dw
dL_db = ...            # dL_dz por dz/db
print(f"z={z}, a={a}, L={L}")

In [ ]:
def revisar_parte2():
    try:
        wt = torch.tensor(0.5, requires_grad=True)
        bt = torch.tensor(-1.0, requires_grad=True)
        Lt = (torch.sigmoid(wt * 2.0 + bt) - 1.0) ** 2
        Lt.backward()                                   # PyTorch recorre el grafo hacia atrás
        print(f"A mano:   dL/dw = {dL_dw:.4f}   dL/db = {dL_db:.4f}")
        print(f"PyTorch:  dL/dw = {wt.grad.item():.4f}   dL/db = {bt.grad.item():.4f}")
        num = derivada_central(lambda v: (sigmoide(v * 2 - 1) - 1) ** 2, 0.5)
        print(f"Numérica: dL/dw = {num:.4f}")
        ok = abs(dL_dw - wt.grad.item()) < 1e-6 and abs(dL_db - bt.grad.item()) < 1e-6
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ regla de la cadena" if ok else "❌ regla de la cadena: dL_dz = dL_da·a(1−a); dL_dw = dL_dz·x; dL_db = dL_dz")
    GLEIHAR["parte2"] = "1/1" if ok else "0/1"
revisar_parte2()

> ⚠️ **Cuidado:** los gradientes se **acumulan**. Ejecuta la celda siguiente: el segundo `backward()` suma otra vez $-0{,}5$.

In [ ]:
wt = torch.tensor(0.5, requires_grad=True)
for vez in [1, 2]:
    L = (torch.sigmoid(wt * 2.0 - 1.0) - 1.0) ** 2
    L.backward()
    print(f"después del backward número {vez}: w.grad = {wt.grad.item()}")
wt.grad.zero_()
print("después de zero_():", wt.grad.item())

## 3️⃣ Gradiente y curvas de nivel (sección 3.3)

$$L(w,b)=\frac1n\sum_i (wx_i+b-y_i)^2,\qquad \nabla L(10,1)=(-7{,}5;\ -3{,}4)$$

In [ ]:
X = torch.tensor([1.0, 1.5, 2.0, 2.5, 3.0])
Y = torch.tensor([11.5, 18.0, 22.5, 28.5, 33.0])
def perdida(w, b):
    return ((w * X + b - Y) ** 2).mean()

p = torch.tensor([10.0, 1.0], requires_grad=True)
Lp = perdida(p[0], p[1]); Lp.backward()
grad = p.grad.clone()
print(f"L(10, 1) = {Lp.item():.3f}    ∇L(10, 1) = {grad.tolist()}")

W, B = torch.meshgrid(torch.linspace(8, 13.5, 200), torch.linspace(-4.5, 7, 200), indexing="ij")
Z = ((W[..., None] * X + B[..., None] - Y) ** 2).mean(-1)
plt.figure(figsize=(5, 6.5))
cs = plt.contour(W, B, Z, levels=[0.3, 0.6, 1.2, 2.2, 3.35, 6, 10], cmap="viridis")
plt.clabel(cs, fontsize=7)
plt.plot(10.7, 1.3, "k*", ms=12, label="mínimo (10,7; 1,3)")
plt.quiver(10, 1, -grad[0], -grad[1], color="tab:orange", angles="xy", scale_units="xy", scale=4, width=0.012, zorder=5, label="−∇L en (10; 1)")
plt.gca().set_aspect("equal"); plt.xlabel("w"); plt.ylabel("b"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.title("Curvas de nivel de la pérdida de los terrenos"); plt.show()
GLEIHAR["parte3"] = {"grad_10_1": [round(v, 3) for v in grad.tolist()]}

> ✅ **Comprueba:** la flecha cruza la curva de nivel $L=3{,}35$ en ángulo recto, pero **no apunta al mínimo**: las elipses son muy alargadas. Es el mal condicionamiento de la sección 3.4 ($\kappa\approx58$).

## 4️⃣ Tu ciclo de entrenamiento (secciones 3.4 y 3.5)

1. Adelante → 2. Pérdida → 3. `backward()` → 4. Paso → 5. Borrar

### ✍️ Tu turno: completa los pasos 3, 4 y 5.

In [ ]:
def entrenar(eta, pasos=300):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    historial = []
    for paso in range(pasos):
        L = perdida(w, b)                 # 1 y 2: adelante y pérdida
        historial.append(L.item())
        ...                               # 3: atrás
        with torch.no_grad():
            ...                           # 4: w ← w − eta·w.grad   y   b ← b − eta·b.grad
        ...                               # 5: borra los gradientes de w y de b
    return w.item(), b.item(), historial

> 🔮 **Predice:** según la sección 3.4, el límite es $\eta<2/\lambda_{\max}\approx0{,}185$. ¿Qué pasará con $\eta=0{,}19$?

In [ ]:
def revisar_parte4():
    try:
        resultados = {}
        plt.figure(figsize=(7, 4))
        for eta in [0.05, 0.1, 0.18, 0.19]:
            w, b, h = entrenar(eta)
            resultados[eta] = (w, b, h[-1])
            plt.semilogy(h, label=f"η = {eta}")
            print(f"η = {eta:<5} → w = {w:9.3f}   b = {b:9.3f}   pérdida final = {h[-1]:.4g}")
        plt.xlabel("paso"); plt.ylabel("pérdida (escala log)"); plt.legend(); plt.grid(alpha=.3); plt.show()
        w18, b18, _ = resultados[0.18]
        ok = abs(w18 - 10.7) < 0.05 and abs(b18 - 1.3) < 0.1 and resultados[0.19][2] > 1e3
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ ciclo de entrenamiento" if ok else "❌ ciclo: L.backward(); luego w -= eta * w.grad (igual b); luego w.grad.zero_() y b.grad.zero_()")
    GLEIHAR["parte4"] = "1/1" if ok else "0/1"
revisar_parte4()

Lo mismo con un **optimizador** de PyTorch, que hace los pasos 4 y 5 por ti:

In [ ]:
w = torch.tensor(0.0, requires_grad=True); b = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.SGD([w, b], lr=0.18)
for paso in range(300):
    L = perdida(w, b)
    L.backward(); opt.step(); opt.zero_grad()
print(f"Con torch.optim.SGD: w = {w.item():.3f}, b = {b.item():.3f}, pérdida = {L.item():.4f}")

## 5️⃣ Probabilidad por simulación (sección 3.6)

**Ley de los grandes números:** la frecuencia se acerca a la probabilidad cuando hay más datos.

In [ ]:
torch.manual_seed(3)
p = 0.7
lanzamientos = (torch.rand(5000) < p).float()        # 1 = cara con probabilidad 0,7
frecuencia = lanzamientos.cumsum(0) / torch.arange(1, 5001)
plt.figure(figsize=(7, 3.5))
plt.semilogx(frecuencia, label="frecuencia de caras"); plt.axhline(p, color="tab:orange", ls="--", label="p = 0,7")
plt.xlabel("número de lanzamientos"); plt.legend(); plt.grid(alpha=.3); plt.show()

muestras = torch.randn(100_000)                       # normal N(0, 1)
for k in [1, 2, 3]:
    print(f"a menos de {k} desviación(es): {(muestras.abs() < k).float().mean():.3%}")

El detector de textos del libro: 20 % de ensayos con IA, detecta el 90 % de ellos y marca por error el 10 % de los escritos por personas. **¿Qué fracción de los marcados fue generada con IA?**

In [ ]:
torch.manual_seed(4)
n = 100_000
generado = torch.rand(n) < 0.20
marcado = torch.where(generado, torch.rand(n) < 0.90, torch.rand(n) < 0.10)
p_sim = (generado & marcado).sum() / marcado.sum()
print(f"Simulación: P(IA | marcado) = {p_sim:.3f}     Bayes: 0,18 / 0,26 = {0.18/0.26:.3f}")
GLEIHAR["parte5"] = {"frecuencia_final": round(frecuencia[-1].item(), 3), "bayes_sim": round(p_sim.item(), 3)}

## 6️⃣ Entropía cruzada y verosimilitud (sección 3.7)

$$L=-\ln q_y=-z_y+\ln\sum_j e^{z_j},\qquad \nabla_{\mathbf z}L=\mathbf q-\mathbf e_y$$

### ✍️ Tu turno: programa la entropía cruzada a partir de los logits.
Pista: `torch.logsumexp(z, dim=0)` calcula $\ln\sum_j e^{z_j}$ sin desbordarse.

In [ ]:
def entropia_cruzada(z, y):
    """z: logits (tensor de K números); y: índice de la respuesta correcta."""
    ...

In [ ]:
def revisar_parte6():
    try:
        z = torch.tensor([2.0, 1.0, 0.0], requires_grad=True)
        L = entropia_cruzada(z, 0)
        L.backward()
        q = torch.softmax(z.detach(), dim=0)
        ref = F.cross_entropy(z.detach()[None], torch.tensor([0]))
        print(f"tu pérdida = {L.item():.4f}    F.cross_entropy = {ref.item():.4f}")
        print("q =", q.numpy().round(3), "   gradiente =", z.grad.numpy().round(3), "   q − e_y =", (q - torch.tensor([1., 0, 0])).numpy().round(3))
        ok = abs(L.item() - 0.4076) < 1e-3 and torch.allclose(z.grad, q - torch.tensor([1., 0, 0]), atol=1e-5)
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ entropía cruzada" if ok else "❌ entropía cruzada: devuelve -z[y] + torch.logsumexp(z, dim=0)")
    GLEIHAR["parte6"] = "1/1" if ok else "0/1"
revisar_parte6()

La verosimilitud de la moneda con 7 caras en 10 lanzamientos: su máximo está en la frecuencia observada.

In [ ]:
ps = torch.linspace(0.01, 0.99, 200)
log_ver = 7 * torch.log(ps) + 3 * torch.log(1 - ps)
plt.figure(figsize=(6, 3.5))
plt.plot(ps, log_ver); plt.axvline(ps[log_ver.argmax()], color="tab:orange", ls="--", label=f"máximo en p ≈ {ps[log_ver.argmax()]:.2f}")
plt.xlabel("p"); plt.ylabel("log-verosimilitud"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 7️⃣ Gleihar aprende con gradiente (sección 3.7)

En la semana 1, Gleihar v0.0 **contaba** letras. Ahora empieza con un **logit por letra**, todos en cero, y aprende con descenso de gradiente minimizando la entropía cruzada sobre el texto. La teoría dice que debe terminar con $\mathbf q=\mathbf f$: las mismas frecuencias que obtenía contando.

In [ ]:
texto = """En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo
de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero,
salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, algún palomino de añadidura los
domingos, consumían las tres partes de su hacienda. El resto della concluían sayo de velarte, calzas de velludo para
las fiestas, con sus pantuflos de lo mesmo, y los días de entresemana se honraba con su vellorí de lo más fino."""
texto = texto.lower().replace("\n", " ")
letras = sorted(set(texto))
y_texto = torch.tensor([letras.index(c) for c in texto])          # cada letra como índice
f = torch.bincount(y_texto, minlength=len(letras)).float() / len(y_texto)   # frecuencias (Gleihar v0.0)

z = torch.zeros(len(letras), requires_grad=True)                  # un logit por letra
opt = torch.optim.SGD([z], lr=5.0)
perdidas = []
for paso in range(500):
    L = F.cross_entropy(z.expand(len(y_texto), -1), y_texto)     # promedio de −ln q(letra real)
    L.backward(); opt.step(); opt.zero_grad()
    perdidas.append(L.item())
q = torch.softmax(z.detach(), dim=0)
H = -(f[f > 0] * f[f > 0].log()).sum()
print(f"pérdida inicial = {perdidas[0]:.4f} (= ln {len(letras)} = {math.log(len(letras)):.4f})")
print(f"pérdida final   = {perdidas[-1]:.4f}    entropía del texto H(f) = {H:.4f}")
print(f"máxima diferencia entre q (gradiente) y f (contar): {(q - f).abs().max():.5f}")
plt.figure(figsize=(9, 3.5))
orden = f.argsort(descending=True)[:15]
plt.bar([repr(letras[i]) for i in orden], f[orden], alpha=.5, label="contando (v0.0)")
plt.plot([repr(letras[i]) for i in orden], q[orden], "o", color="tab:orange", label="con gradiente")
plt.legend(); plt.title("Las 15 letras más frecuentes"); plt.show()

> ✅ **Comprueba:** las dos versiones coinciden. Contar **era** máxima verosimilitud; el gradiente llega al mismo lugar. La diferencia es que el gradiente sirve para **cualquier** modelo: en la semana 6, los logits los calculará una red neuronal que mira las letras anteriores.

Ahora Gleihar escribe 80 letras muestreando de $\mathbf q$:

In [ ]:
torch.manual_seed(2026)
muestra = "".join(letras[i] for i in torch.multinomial(q, 80, replacement=True))
print(f"«{muestra}»")
GLEIHAR["parte7"] = {"perdida_final": round(perdidas[-1], 4), "entropia": round(H.item(), 4),
                     "dif_max_q_f": round((q - f).abs().max().item(), 5), "muestra": muestra[:40]}

> 🔮 Gleihar acierta las proporciones de las letras, pero el texto no se parece al español. ¿Por qué? (Pista: el ejemplo «ana» de la sección 3.6.)

## 8️⃣ Tu reporte

Copia el resultado y pégalo en el cuadro de texto de la **Tarea 3**.

In [ ]:
GLEIHAR["fecha"] = time.strftime("%Y-%m-%d %H:%M")
with open("reporte_semana03.json", "w", encoding="utf-8") as fh:
    json.dump(GLEIHAR, fh, ensure_ascii=False, indent=2)
print(json.dumps(GLEIHAR, ensure_ascii=False, indent=2))
print("\n✅ Copia este reporte y pégalo en la Tarea 3.")

---

## 🏁 Lo que aprendiste hoy

- ✔️ La **diferencia central** aproxima derivadas con error $\approx h^2$ y sirve para comprobar gradientes.
- ✔️ La **regla de la cadena** hacia atrás es lo que hace `backward()`.
- ✔️ El **gradiente** es perpendicular a las curvas de nivel y apunta cuesta arriba.
- ✔️ La **tasa de aprendizaje** tiene un límite: por encima de $2/\lambda_{\max}$ el entrenamiento explota.
- ✔️ La **entropía cruzada** sale de la máxima verosimilitud, y Gleihar ya **aprende con gradiente**.

### 🔜 La próxima semana

**Datos y tokenización:** cómo se limpia un texto, cómo se parte en tokens y cómo se prepara un conjunto de datos para entrenar a Gleihar.